In [12]:
import urllib.parse
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup
import io

In [3]:
# Define custom User-Agent header (required by Wikipedia API/scraping policies)
headers = {
    'User-Agent': 'WinterXGamesScraper/1.0 (ross.bennfors@theinformationlab.co.uk)'
}

main_url = "https://en.wikipedia.org/wiki/X_Games"

In [4]:
print("1. Fetching main X Games index page...")
response = requests.get(main_url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

# Discover all individual Winter X Games event page links
xgames_urls = []
for a_tag in soup.find_all('a', href=True):
    href = a_tag['href']
    if '/wiki/Winter_X_Games' in href and 'action=edit' not in href:
        full_url = urllib.parse.urljoin("https://en.wikipedia.org", href)
        if full_url not in xgames_urls:
            xgames_urls.append(full_url)

# --- VERIFICATION CHECKS ---
print(f"\nSUCCESS: Collected {len(xgames_urls)} event URLs!\n")


1. Fetching main X Games index page...

SUCCESS: Collected 33 event URLs!



In [5]:
print("--- URL LIST PREVIEW (First 5) ---")
for i, url in enumerate(xgames_urls[:5], start=1):
    print(f"[{i}] {url}")

if xgames_urls:
    sample_url = xgames_urls[0]
    print(f"\n2. Testing table extraction from sample page: {sample_url}")
    
    # Read all HTML tables on the page into a list of DataFrames
    tables = pd.read_html(sample_url, storage_options=headers)
    
    print(f"\nTotal tables found on this page: {len(tables)}\n")
    
    # Print out summary details for every table on the page
    for index, df in enumerate(tables):
        print(f"================ TABLE INDEX [{index}] ================")
        print("Dimensions (Rows, Columns):", df.shape)
        print("Column Headers:", df.columns.tolist()[:6])  # Show up to 6 headers
        print("\nFirst 2 rows preview:")
        print(df.head(2))
        print("\n")

--- URL LIST PREVIEW (First 5) ---
[1] https://en.wikipedia.org/wiki/Winter_X_Games_IX
[2] https://en.wikipedia.org/wiki/Winter_X_Games_X
[3] https://en.wikipedia.org/wiki/Winter_X_Games_XI
[4] https://en.wikipedia.org/wiki/Winter_X_Games_XII
[5] https://en.wikipedia.org/wiki/Winter_X_Games_XIII

2. Testing table extraction from sample page: https://en.wikipedia.org/wiki/Winter_X_Games_IX

Total tables found on this page: 15

================ TABLE INDEX [0] ================
Dimensions (Rows, Columns): (3, 3)
Column Headers: ['Place', 'Athlete', 'Score']

First 2 rows preview:
    Place       Athlete  Score
0    Gold  Brian Deegan   93.0
1  Silver  Jeff Kargola   92.2


================ TABLE INDEX [1] ================
Dimensions (Rows, Columns): (3, 3)
Column Headers: ['Place', 'Athlete', 'Time']

First 2 rows preview:
    Place       Athlete  Time
0    Gold  Reggie Crist   NaN
1  Silver    Zach Crist   NaN


================ TABLE INDEX [2] ================
Dimensions (Rows, Columns)

In [6]:
print("--- URL LIST PREVIEW ---")
for i, url in enumerate(xgames_urls, start=1):
    print(f"[{i}] {url}")

--- URL LIST PREVIEW ---
[1] https://en.wikipedia.org/wiki/Winter_X_Games_IX
[2] https://en.wikipedia.org/wiki/Winter_X_Games_X
[3] https://en.wikipedia.org/wiki/Winter_X_Games_XI
[4] https://en.wikipedia.org/wiki/Winter_X_Games_XII
[5] https://en.wikipedia.org/wiki/Winter_X_Games_XIII
[6] https://en.wikipedia.org/wiki/Winter_X_Games_XIV
[7] https://en.wikipedia.org/wiki/Winter_X_Games_XV
[8] https://en.wikipedia.org/wiki/Winter_X_Games_XVI
[9] https://en.wikipedia.org/wiki/Winter_X_Games_XVII
[10] https://en.wikipedia.org/wiki/Winter_X_Games_XVIII
[11] https://en.wikipedia.org/wiki/Winter_X_Games_XIX
[12] https://en.wikipedia.org/wiki/Winter_X_Games_XX
[13] https://en.wikipedia.org/wiki/Winter_X_Games_XXI
[14] https://en.wikipedia.org/wiki/Winter_X_Games_XXII
[15] https://en.wikipedia.org/wiki/Winter_X_Games_XXIII
[16] https://en.wikipedia.org/wiki/Winter_X_Games_XXIV
[17] https://en.wikipedia.org/wiki/Winter_X_Games_XXV
[18] https://en.wikipedia.org/wiki/Winter_X_Games_XXVI
[19] http

In [7]:
xgames_urls = xgames_urls[:20] # Keep first 20

In [8]:
sample_url = xgames_urls[0]
print(f"Extracting tables from: {sample_url}")

# Fetch all tables on the page
tables = pd.read_html(sample_url, storage_options=headers)

print(f"Successfully retrieved {len(tables)} tables from page 1.\n")

Extracting tables from: https://en.wikipedia.org/wiki/Winter_X_Games_IX
Successfully retrieved 15 tables from page 1.



In [9]:
for index, df in enumerate(tables):
    print(f"******************** START OF TABLE INDEX [{index}] ********************")
    print(f"Table Shape: {df.shape[0]} rows x {df.shape[1]} columns\n")
    
    # Setting display options so pandas doesn't truncate/cut off columns or rows
    with pd.option_context('display.max_columns', None), pd.option_context('display.max_rows', None):
        print(df)
        
    print(f"********************* END OF TABLE INDEX [{index}] *********************\n\n")

******************** START OF TABLE INDEX [0] ********************
Table Shape: 3 rows x 3 columns

    Place        Athlete  Score
0    Gold   Brian Deegan   93.0
1  Silver   Jeff Kargola   92.2
2  Bronze  Dustin Miller   91.6
********************* END OF TABLE INDEX [0] *********************


******************** START OF TABLE INDEX [1] ********************
Table Shape: 3 rows x 3 columns

    Place        Athlete  Time
0    Gold   Reggie Crist   NaN
1  Silver     Zach Crist   NaN
2  Bronze  Enak Gavaggio   NaN
********************* END OF TABLE INDEX [1] *********************


******************** START OF TABLE INDEX [2] ********************
Table Shape: 3 rows x 3 columns

    Place            Athlete    Time
0    Gold    Sanna Tidstrand  55.097
1  Silver      Karin Huttary  55.385
2  Bronze  Magdalena Jonsson  55.662
********************* END OF TABLE INDEX [2] *********************


******************** START OF TABLE INDEX [3] ********************
Table Shape: 3 rows x 3 co

In [16]:
# Target the first edition page in your list
sample_url = xgames_urls[0]
print(f"Extracting section titles and tables from event page:\n{sample_url}\n")

response2 = requests.get(sample_url, headers=headers)
soup2 = BeautifulSoup(response2.text, 'html.parser')

tables_with_titles = []

# Find all table elements in the Wikipedia article body
for table_tag in soup2.find_all('table'):
    
    # Search backwards for the nearest preceding heading tag (h2, h3, or h4)
    heading_tag = table_tag.find_previous(['h2', 'h3', 'h4'])
    
    # Clean the heading text (remove [edit] tags if present)
    if heading_tag:
        title_text = heading_tag.get_text().replace('[edit]', '').strip()
    else:
        title_text = "General / Overview"
        
    # Convert the individual HTML table tag into a pandas DataFrame
    # io.StringIO converts string HTML directly into pandas readable stream
    try:
        df_list = pd.read_html(io.StringIO(str(table_tag)))
        if df_list:
            df = df_list[0]
            
            # Attach the extracted title and source URL to the table!
            df['Event_Title'] = title_text
            df['Source_URL'] = sample_url
            
            tables_with_titles.append((title_text, df))
    except Exception:
        continue  # Skip invalid or empty layout tables

print(f"Successfully extracted {len(tables_with_titles)} tables with section headings!\n")

# Preview the results
for index, (title, df) in enumerate(tables_with_titles):
    print(f"--- TABLE INDEX [{index}] | HEADING: '{title}' ---")
    print(f"Shape: {df.shape}")
    print("Columns:", df.columns.tolist()[:5])
    print(df.head(2))
    print("\n" + "="*60 + "\n")

Extracting section titles and tables from event page:
https://en.wikipedia.org/wiki/Winter_X_Games_IX

Successfully extracted 15 tables with section headings!

--- TABLE INDEX [0] | HEADING: 'Motocross' ---
Shape: (3, 5)
Columns: ['Place', 'Athlete', 'Score', 'Event_Title', 'Source_URL']
    Place       Athlete  Score Event_Title  \
0    Gold  Brian Deegan   93.0   Motocross   
1  Silver  Jeff Kargola   92.2   Motocross   

                                        Source_URL  
0  https://en.wikipedia.org/wiki/Winter_X_Games_IX  
1  https://en.wikipedia.org/wiki/Winter_X_Games_IX  


--- TABLE INDEX [1] | HEADING: 'Skiing' ---
Shape: (3, 5)
Columns: ['Place', 'Athlete', 'Time', 'Event_Title', 'Source_URL']
    Place       Athlete  Time Event_Title  \
0    Gold  Reggie Crist   NaN      Skiing   
1  Silver    Zach Crist   NaN      Skiing   

                                        Source_URL  
0  https://en.wikipedia.org/wiki/Winter_X_Games_IX  
1  https://en.wikipedia.org/wiki/Winter_X_Ga

In [18]:
tables_with_titles 

[('Motocross',
      Place        Athlete  Score Event_Title  \
  0    Gold   Brian Deegan   93.0   Motocross   
  1  Silver   Jeff Kargola   92.2   Motocross   
  2  Bronze  Dustin Miller   91.6   Motocross   
  
                                          Source_URL  
  0  https://en.wikipedia.org/wiki/Winter_X_Games_IX  
  1  https://en.wikipedia.org/wiki/Winter_X_Games_IX  
  2  https://en.wikipedia.org/wiki/Winter_X_Games_IX  ),
 ('Skiing',
      Place        Athlete  Time Event_Title  \
  0    Gold   Reggie Crist   NaN      Skiing   
  1  Silver     Zach Crist   NaN      Skiing   
  2  Bronze  Enak Gavaggio   NaN      Skiing   
  
                                          Source_URL  
  0  https://en.wikipedia.org/wiki/Winter_X_Games_IX  
  1  https://en.wikipedia.org/wiki/Winter_X_Games_IX  
  2  https://en.wikipedia.org/wiki/Winter_X_Games_IX  ),
 ('Skiing',
      Place            Athlete    Time Event_Title  \
  0    Gold    Sanna Tidstrand  55.097      Skiing   
  1  Silver    

In [19]:
all_event_tables = []

print(f"Starting extraction across {len(xgames_urls)} event pages...\n")

for index, url in enumerate(xgames_urls, start=1):
    print(f"[{index}/{len(xgames_urls)}] Fetching: {url}")
    
    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        page_tables_count = 0
        
        # Loop through every table element on the event page
        for table_tag in soup.find_all('table'):
            
            # Find the closest preceding section heading (h2, h3, or h4)
            heading_tag = table_tag.find_previous(['h2', 'h3', 'h4'])
            
            if heading_tag:
                title_text = heading_tag.get_text().replace('[edit]', '').strip()
            else:
                title_text = "General / Overview"
            
            # Skip Wikipedia navigation templates or infoboxes if necessary
            # Filter for event/results headings
            try:
                df_list = pd.read_html(io.StringIO(str(table_tag)))
                if df_list:
                    df = df_list[0]
                    
                    # Flatten multi-level column headers if present
                    if isinstance(df.columns, pd.MultiIndex):
                        df.columns = ['_'.join(col).strip() for col in df.columns.values]
                    
                    # Attach metadata columns
                    df['Event_Title'] = title_text
                    df['Source_URL'] = url
                    
                    all_event_tables.append(df)
                    page_tables_count += 1
            except Exception:
                continue  # Skip unparseable tables
                
        print(f"   --> Extracted {page_tables_count} tables from this page.")
        
        # Polite delay to respect Wikipedia servers
        time.sleep(5)
        
    except Exception as e:
        print(f"   --> Error fetching {url}: {e}")

Starting extraction across 20 event pages...

[1/20] Fetching: https://en.wikipedia.org/wiki/Winter_X_Games_IX
   --> Extracted 15 tables from this page.
[2/20] Fetching: https://en.wikipedia.org/wiki/Winter_X_Games_X
   --> Extracted 30 tables from this page.
[3/20] Fetching: https://en.wikipedia.org/wiki/Winter_X_Games_XI
   --> Extracted 17 tables from this page.
[4/20] Fetching: https://en.wikipedia.org/wiki/Winter_X_Games_XII
   --> Extracted 19 tables from this page.
[5/20] Fetching: https://en.wikipedia.org/wiki/Winter_X_Games_XIII
   --> Extracted 21 tables from this page.
[6/20] Fetching: https://en.wikipedia.org/wiki/Winter_X_Games_XIV
   --> Extracted 28 tables from this page.
[7/20] Fetching: https://en.wikipedia.org/wiki/Winter_X_Games_XV
   --> Extracted 31 tables from this page.
[8/20] Fetching: https://en.wikipedia.org/wiki/Winter_X_Games_XVI
   --> Extracted 22 tables from this page.
[9/20] Fetching: https://en.wikipedia.org/wiki/Winter_X_Games_XVII
   --> Extracted 21